# Example analysis: three sources, one catalog

A worked tour of the astrolabe pipeline, live against the real services:

1. **Gaia DR3** — cone search on the Pleiades, persisted through the `Store`,
   then an HR (colour–magnitude) diagram of the cluster members.
2. **Gaia × SDSS** — both surveys queried on the same equatorial field, cross-matched
   positionally at 1 arcsec, and the photometry compared.
3. **JPL Horizons** — a year of Mars ephemerides, plotted as a sky track (watch for
   the retrograde loop) and a distance/brightness curve.

Everything fetched lands in `data/` via `astrolabe.store.Store` (Parquet + sidecar
metadata), and the last section queries it all back with DuckDB SQL. Requires network;
a full run takes a couple of minutes, dominated by the Gaia TAP job.

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from astrolabe.analysis import crossmatch, hr_diagram
from astrolabe.sources import get_source
from astrolabe.store import Store

# Root the store at the repo's data/ directory regardless of where the kernel runs.
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
store = Store(root / "data")
store.data_dir

## 1. Gaia — the Pleiades

A 1.5° cone around the cluster centre. The adapter builds the ADQL, runs it against
the ESA TAP service, and hands back an astropy `Table` already carrying the house
standard columns (`source_id`, `ra`, `dec`) plus Gaia photometry and parallax.

In [ ]:
PLEIADES = {"ra": 56.75, "dec": 24.1167, "radius": 1.5, "limit": 8000}

gaia = get_source("gaia")
pleiades = gaia.query(PLEIADES)
print(f"{len(pleiades)} rows")
pleiades[:5]

In [ ]:
meta = store.write(pleiades, name="pleiades_gaia", source="gaia", query=PLEIADES)
meta

### HR diagram

The Pleiades sit at ~136 pc (parallax ≈ 7.4 mas), so a parallax window separates
cluster members from the fore/background. Left: every star in the field with a
usable parallax. Right: the members only — the cluster's main sequence pops out.

In [ ]:
parallax = np.asarray(pleiades["parallax"], dtype=float)
member_mask = (parallax > 6.5) & (parallax < 8.5)
members = pleiades[member_mask]
print(f"{member_mask.sum()} of {len(pleiades)} stars in the parallax window (6.5–8.5 mas)")

fig_field = hr_diagram(pleiades)
fig_field.axes[0].set_title("HR diagram — full field")
fig_field

In [ ]:
fig_members = hr_diagram(members)
fig_members.axes[0].set_title("HR diagram — Pleiades members (parallax cut)")
fig_members

## 2. Gaia × SDSS cross-match

The Pleiades are outside the SDSS footprint, so for the cross-match we pick a small
field on the celestial equator (well inside SDSS coverage) and query **both** surveys
over it. `crossmatch` then pairs each Gaia star with its nearest SDSS object within
1 arcsec — the standard sanity check that two catalogs are talking about the same sky.

> **Workaround (2026-07):** astroquery's SDSS module currently gets an HTML error page
> back from SkyServer's `x_results.aspx` endpoint (both 0.4.11 and dev), while
> SkyServer's REST API (`SkyServerWS/SearchTools/SqlSearch`) works fine. The adapter's
> `_run_sql` is the designed injection seam (tests use it for fixtures), so we point it
> at the working REST endpoint here — the data is still live SDSS DR17, and the
> adapter's normalization (`objid -> source_id`) still runs.

In [ ]:
import requests
from astropy.table import Table


def _skyserver_sql(sql: str) -> Table:
    """Live SDSS DR17 SQL via the SkyServer REST API (see workaround note above)."""
    resp = requests.get(
        "https://skyserver.sdss.org/dr17/SkyServerWS/SearchTools/SqlSearch",
        params={"cmd": sql, "format": "csv"},
        timeout=60,
    )
    resp.raise_for_status()
    return Table.read(resp.text, format="ascii.csv", comment="#")


sdss = get_source("sdss")
sdss._run_sql = _skyserver_sql  # inject the working fetcher through the seam

In [ ]:
FIELD = {"ra": 180.0, "dec": 0.0, "radius": 0.02}  # ~72 arcsec cone on the equator

sdss_field = sdss.query(
    {
        "sql": f"""
        SELECT p.objid, p.ra, p.dec, p.u, p.g, p.r, p.i, p.z
        FROM PhotoObj p
        JOIN dbo.fGetNearbyObjEq({FIELD["ra"]}, {FIELD["dec"]}, {FIELD["radius"] * 60}) n
          ON p.objID = n.objID
        """
    }
)
gaia_field = gaia.query({**FIELD, "limit": 2000})
print(f"SDSS: {len(sdss_field)} objects, Gaia: {len(gaia_field)} objects")
sdss_field[:5]

In [ ]:
matched = crossmatch(gaia_field, sdss_field, radius_arcsec=1.0)
print(f"{len(matched)} Gaia stars matched to an SDSS object within 1 arcsec")
matched[:5]

In [ ]:
store.write(sdss_field, name="field_eq180_sdss", source="sdss", query=FIELD)
store.write(
    matched,
    name="field_eq180_xmatch",
    source="gaia+sdss",
    query={**FIELD, "radius_arcsec": 1.0},
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.hist(matched["sep_arcsec"], bins=25, color="tab:blue", alpha=0.8)
ax1.set_xlabel("separation (arcsec)")
ax1.set_ylabel("pairs")
ax1.set_title("Match separations")

g = np.asarray(matched["phot_g_mean_mag"], dtype=float)
r = np.asarray(matched["r"], dtype=float)
ax2.scatter(r, g, s=12, alpha=0.7, edgecolors="none")
lo, hi = np.nanmin(r) - 0.5, np.nanmax(r) + 0.5
ax2.plot([lo, hi], [lo, hi], ls="--", c="gray", lw=1, label="G = r")
ax2.set_xlabel("SDSS r (mag)")
ax2.set_ylabel("Gaia G (mag)")
ax2.set_title("Photometry of matched pairs")
ax2.invert_xaxis()
ax2.invert_yaxis()
ax2.legend()
fig.tight_layout()

The separation histogram should pile up well below 1 arcsec (real matches), and Gaia G
tracks SDSS r closely since the G band broadly overlaps r — points scatter around the
G = r line with colour-dependent offsets.

## 3. JPL Horizons — a year of Mars

Ephemerides are time-series rather than a static catalog, so the adapter is thin: it
returns Horizons' table normalized to carry `ra`/`dec` and a `source_id` equal to the
target id. We pull geocentric positions for Mars through 2026 at a 3-day cadence.

In [ ]:
MARS_QUERY = {
    "target": "499",  # Mars barycenter
    "location": "500@399",  # geocentric
    "epochs": {"start": "2026-01-01", "stop": "2026-12-31", "step": "3d"},
}

horizons = get_source("horizons")
mars = horizons.query(MARS_QUERY)
print(f"{len(mars)} epochs, columns include: {mars.colnames[:12]} ...")

# Keep a tidy subset for storage: position, brightness, and distances.
mars_slim = mars["source_id", "datetime_str", "datetime_jd", "ra", "dec", "V", "delta", "r"]
store.write(mars_slim, name="mars_ephem_2026", source="horizons", query=MARS_QUERY)
mars_slim[:5]

In [ ]:
jd = np.asarray(mars_slim["datetime_jd"], dtype=float)
days = jd - jd[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

sc = ax1.scatter(mars_slim["ra"], mars_slim["dec"], c=days, cmap="viridis", s=14)
ax1.set_xlabel("RA (deg)")
ax1.set_ylabel("Dec (deg)")
ax1.set_title("Mars sky track, 2026 (geocentric)")
ax1.invert_xaxis()  # sky convention: RA increases to the left
fig.colorbar(sc, ax=ax1, label="days since 2026-01-01")

ax2.plot(days, mars_slim["delta"], c="tab:red", label="Earth–Mars distance (au)")
ax2.set_xlabel("days since 2026-01-01")
ax2.set_ylabel("distance (au)", color="tab:red")
ax2b = ax2.twinx()
ax2b.plot(days, mars_slim["V"], c="tab:blue", label="V magnitude")
ax2b.set_ylabel("apparent V mag", color="tab:blue")
ax2b.invert_yaxis()  # brighter = up
ax2.set_title("Distance and brightness")
fig.tight_layout()

As Earth gains on Mars ahead of the early-2027 opposition, the Earth–Mars distance
falls through the year and Mars brightens accordingly — the two curves mirror each
other by construction ($V \propto 5\log_{10}\Delta$ plus phase effects).

## 4. Query it all back with SQL

Every dataset written above is now a Parquet file under `data/processed/` with a JSON
sidecar, and `Store.query` exposes each one as a DuckDB view named after it.

In [ ]:
for name in store.list_datasets():
    m = store.read_meta(name)
    print(f"{name:24s} {m.n_rows:6d} rows  from {m.source:10s} fetched {m.fetched_at[:19]}")

In [ ]:
# The ten brightest Pleiades members, straight off the stored Parquet.
store.query("""
    SELECT source_id, phot_g_mean_mag, bp_rp, parallax
    FROM pleiades_gaia
    WHERE parallax BETWEEN 6.5 AND 8.5
    ORDER BY phot_g_mean_mag
    LIMIT 10
""")

In [ ]:
# One aggregate across each dataset — the catalog is source-agnostic at this layer.
store.query("""
    SELECT 'pleiades_gaia' AS dataset, count(*) AS n, round(avg(ra), 3) AS mean_ra
    FROM pleiades_gaia
    UNION ALL
    SELECT 'field_eq180_xmatch', count(*), round(avg(ra), 3) FROM field_eq180_xmatch
    UNION ALL
    SELECT 'mars_ephem_2026', count(*), round(avg(ra), 3) FROM mars_ephem_2026
""")

## Wrap-up

- Three very different providers (TAP/ADQL, SkyServer, Horizons) all arrive as astropy
  Tables with the same `source_id`/`ra`/`dec` spine — the `Source` seam doing its job.
- `Store.write` made each fetch a self-describing dataset (Parquet + metadata sidecar);
  `Store.query` gives ad-hoc SQL over all of them with zero setup.
- The analysis layer stayed pure: `crossmatch` and `hr_diagram` never touched the
  network or the filesystem.

Per SPEC §4, anything here that stabilizes into a reusable pattern should be promoted
out of the notebook into `src/astrolabe/`.